# Piotr Korcz PDD1

## Importy

In [ ]:
import os
import random
import time

import pandas as pd
import matplotlib.pyplot as plt

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    LongType,
    StringType,
)

## Spark na klastrze i ścieżki HDFS

In [ ]:
spark = (
    SparkSession.builder
    .master("spark://master:7077")
    .appName("PDD14_MovieMatchLSH")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

RATINGS_PATH = "hdfs://master:9000/movielens/ml-32m/ratings.csv"
MOVIES_PATH = "hdfs://master:9000/movielens/ml-32m/movies.csv"
OUTPUT_PATH = "hdfs://master:9000/movielens/pdd14_out"

NUM_PARTITIONS = 200
NUM_BANDS = 20
ROWS_PER_BAND = 5
SIGNATURE_LENGTH = NUM_BANDS * ROWS_PER_BAND
THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.7]
PRIME = 4294967311
SEED = 10

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

## Wczytanie danych

In [ ]:
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True),
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

ratings_df = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(ratings_schema)
    .csv(RATINGS_PATH)
    .repartition(NUM_PARTITIONS, "userId")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

movies_df = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(movies_schema)
    .csv(MOVIES_PATH)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

ratings_df.createOrReplaceTempView("ratings")
movies_df.createOrReplaceTempView("movies")

print("ratings rows:", ratings_df.count())
print("movies rows:", movies_df.count())

## Funkcje pomocnicze do pokazywania i zapisywania wyników

In [ ]:
def show_and_save(df, name, n=20):
    path = f"{OUTPUT_PATH}/{name}"
    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(path)
    )
    print(name)
    df.show(n, truncate=False)
    return df

## Widoki pomocnicze

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW ratings_enriched AS
SELECT
    r.userId,
    r.movieId,
    r.rating,
    r.timestamp,
    m.title,
    m.genres,
    year(from_unixtime(r.timestamp)) AS rating_year
FROM ratings r
JOIN movies m
ON r.movieId = m.movieId
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW movie_genres AS
SELECT
    movieId,
    title,
    explode(split(genres, '\\\\|')) AS genre
FROM movies
WHERE genres <> '(no genres listed)'
""")

genres_count_df = spark.sql("""
SELECT COUNT(DISTINCT genre) AS distinct_genres_count
FROM movie_genres
""")

show_and_save(genres_count_df, "00_genres_count")

## 1. Liczba unikalnych użytkowników, filmów i wszystkich ocen

In [ ]:
dataset_summary_df = spark.sql("""
SELECT
    COUNT(DISTINCT userId) AS unique_users,
    COUNT(DISTINCT movieId) AS unique_rated_movies,
    (SELECT COUNT(*) FROM movies) AS total_movies_in_catalog,
    COUNT(*) AS total_ratings
FROM ratings
""")

show_and_save(dataset_summary_df, "01_dataset_summary")

## 2. Średnie oceny filmów i użytkowników oraz histogramy

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW movie_avg_ratings AS
SELECT
    r.movieId,
    m.title,
    AVG(r.rating) AS avg_movie_rating,
    COUNT(*) AS movie_rating_count
FROM ratings r
JOIN movies m
ON r.movieId = m.movieId
GROUP BY r.movieId, m.title
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW user_avg_ratings AS
SELECT
    userId,
    AVG(rating) AS avg_user_rating,
    COUNT(*) AS user_rating_count
FROM ratings
GROUP BY userId
""")

movie_avg_hist_df = spark.sql("""
SELECT
    CAST(FLOOR(avg_movie_rating / 0.25) * 0.25 AS DOUBLE) AS bin_start,
    COUNT(*) AS count
FROM movie_avg_ratings
GROUP BY CAST(FLOOR(avg_movie_rating / 0.25) * 0.25 AS DOUBLE)
ORDER BY bin_start
""")

user_avg_hist_df = spark.sql("""
SELECT
    CAST(FLOOR(avg_user_rating / 0.25) * 0.25 AS DOUBLE) AS bin_start,
    COUNT(*) AS count
FROM user_avg_ratings
GROUP BY CAST(FLOOR(avg_user_rating / 0.25) * 0.25 AS DOUBLE)
ORDER BY bin_start
""")

show_and_save(movie_avg_hist_df, "02_movie_avg_histogram_bins", n=100)
show_and_save(user_avg_hist_df, "03_user_avg_histogram_bins", n=100)

movie_hist_pd = movie_avg_hist_df.toPandas()
user_hist_pd = user_avg_hist_df.toPandas()

plt.figure(figsize=(10, 5))
plt.bar(movie_hist_pd["bin_start"], movie_hist_pd["count"], width=0.23)
plt.title("Histogram średnich ocen filmów")
plt.xlabel("Średnia ocena filmu")
plt.ylabel("Liczba filmów")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(user_hist_pd["bin_start"], user_hist_pd["count"], width=0.23)
plt.title("Histogram średnich ocen użytkowników")
plt.xlabel("Średnia ocena użytkownika")
plt.ylabel("Liczba użytkowników")
plt.grid(True)
plt.show()

## 3. Średnia ocen filmów i średnia ocen użytkowników

In [ ]:
average_summary_df = spark.sql("""
SELECT
    ROUND((SELECT AVG(avg_movie_rating) FROM movie_avg_ratings), 6) AS avg_of_movie_averages,
    ROUND((SELECT AVG(avg_user_rating) FROM user_avg_ratings), 6) AS avg_of_user_averages,
    ROUND((SELECT AVG(rating) FROM ratings), 6) AS global_avg_rating
""")

show_and_save(average_summary_df, "04_average_summary")

## 4. Liczba ocen w podziale na lata i rok największej aktywności

In [ ]:
ratings_by_year_df = spark.sql("""
SELECT
    rating_year,
    COUNT(*) AS ratings_count
FROM ratings_enriched
WHERE rating_year IS NOT NULL
GROUP BY rating_year
ORDER BY rating_year
""")

show_and_save(ratings_by_year_df, "05_ratings_by_year", n=100)

most_active_year_df = spark.sql("""
SELECT
    rating_year,
    COUNT(*) AS ratings_count
FROM ratings_enriched
WHERE rating_year IS NOT NULL
GROUP BY rating_year
ORDER BY ratings_count DESC, rating_year ASC
LIMIT 1
""")

show_and_save(most_active_year_df, "06_most_active_year")

ratings_by_year_pd = ratings_by_year_df.toPandas()

plt.figure(figsize=(12, 6))
plt.bar(ratings_by_year_pd["rating_year"], ratings_by_year_pd["ratings_count"])
plt.title("Całkowita liczba ocen w podziale na lata")
plt.xlabel("Rok")
plt.ylabel("Liczba ocen")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

## 5. Średnia ocena gatunków i 5 najlepszych gatunków

In [ ]:
genre_avg_ratings_df = spark.sql("""
SELECT
    mg.genre,
    AVG(r.rating) AS avg_genre_rating,
    COUNT(*) AS ratings_count,
    COUNT(DISTINCT r.movieId) AS movies_count
FROM ratings r
JOIN movie_genres mg
ON r.movieId = mg.movieId
WHERE mg.genre <> '(no genres listed)'
GROUP BY mg.genre
ORDER BY avg_genre_rating DESC, ratings_count DESC, genre ASC
""")

show_and_save(genre_avg_ratings_df, "07_genre_avg_ratings", n=100)

top5_genres_df = spark.sql("""
SELECT
    mg.genre,
    ROUND(AVG(r.rating), 4) AS avg_genre_rating,
    COUNT(*) AS ratings_count,
    COUNT(DISTINCT r.movieId) AS movies_count
FROM ratings r
JOIN movie_genres mg
ON r.movieId = mg.movieId
WHERE mg.genre <> '(no genres listed)'
GROUP BY mg.genre
ORDER BY avg_genre_rating DESC, ratings_count DESC, genre ASC
LIMIT 5
""")

show_and_save(top5_genres_df, "08_top5_genres_by_avg_rating")

## 6. Aktywni użytkownicy i ich 3 najczęściej oceniane gatunki

In [ ]:
active_users_df = spark.sql("""
SELECT
    userId,
    COUNT(*) AS ratings_count
FROM ratings
GROUP BY userId
HAVING COUNT(*) > 1000
ORDER BY ratings_count DESC, userId ASC
""")

active_users_df.createOrReplaceTempView("active_users")

show_and_save(active_users_df, "09_active_users_over_1000", n=50)

active_user_ids_df = spark.sql("""
SELECT userId
FROM active_users
""")

active_user_ids_df.createOrReplaceTempView("active_user_ids")

active_user_top3_genres_df = spark.sql("""
WITH active_user_genre_counts AS (
    SELECT
        r.userId,
        mg.genre,
        COUNT(*) AS genre_rating_count
    FROM ratings r
    JOIN active_user_ids au
    ON r.userId = au.userId
    JOIN movie_genres mg
    ON r.movieId = mg.movieId
    WHERE mg.genre <> '(no genres listed)'
    GROUP BY r.userId, mg.genre
),
ranked AS (
    SELECT
        userId,
        genre,
        genre_rating_count,
        ROW_NUMBER() OVER (
            PARTITION BY userId
            ORDER BY genre_rating_count DESC, genre ASC
        ) AS genre_rank
    FROM active_user_genre_counts
)
SELECT
    userId,
    genre_rank,
    genre,
    genre_rating_count
FROM ranked
WHERE genre_rank <= 3
ORDER BY userId ASC, genre_rank ASC
""")

active_user_top3_genres_df.createOrReplaceTempView("active_user_top3_genres")

show_and_save(active_user_top3_genres_df, "10_active_users_top3_genres")

active_user_report_control_df = spark.sql("""
SELECT
    (SELECT COUNT(*) FROM active_users) AS active_users_count,
    (SELECT COUNT(DISTINCT userId) FROM active_user_top3_genres) AS users_with_top3_report,
    (SELECT COUNT(*) FROM active_user_top3_genres) AS rows_in_top3_report
""")

show_and_save(active_user_report_control_df, "11_active_user_report_control")

## 7. Filtrowanie danych do LSH

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW valid_movies AS
SELECT
    movieId
FROM ratings
GROUP BY movieId
HAVING COUNT(*) >= 50
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW ratings_after_movie_filter AS
SELECT
    r.userId,
    r.movieId,
    r.rating
FROM ratings r
JOIN valid_movies vm
ON r.movieId = vm.movieId
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW valid_users AS
SELECT
    userId
FROM ratings_after_movie_filter
GROUP BY userId
HAVING COUNT(*) >= 20
""")

filtered_ratings_df = spark.sql("""
SELECT
    rmf.userId,
    rmf.movieId,
    rmf.rating
FROM ratings_after_movie_filter rmf
JOIN valid_users vu
ON rmf.userId = vu.userId
""").repartition(NUM_PARTITIONS, "userId").persist(StorageLevel.MEMORY_AND_DISK)

filtered_ratings_df.count()
filtered_ratings_df.createOrReplaceTempView("filtered_ratings")

filter_counts_df = spark.sql("""
SELECT
    (SELECT COUNT(*) FROM valid_movies) AS movies_after_min_50_ratings,
    (SELECT COUNT(*) FROM valid_users) AS users_after_min_20_ratings,
    (SELECT COUNT(*) FROM filtered_ratings) AS ratings_after_filtering
""")

show_and_save(filter_counts_df, "12_lsh_filter_counts")

## 8. Zbiory filmów lubianych i nielubianych

In [ ]:
liked_df = spark.sql("""
SELECT
    userId,
    COLLECT_SET(movieId) AS movies,
    SIZE(COLLECT_SET(movieId)) AS set_size
FROM filtered_ratings
WHERE rating >= 4.0
GROUP BY userId
HAVING SIZE(COLLECT_SET(movieId)) > 0
""").persist(StorageLevel.MEMORY_AND_DISK)

disliked_df = spark.sql("""
SELECT
    userId,
    COLLECT_SET(movieId) AS movies,
    SIZE(COLLECT_SET(movieId)) AS set_size
FROM filtered_ratings
WHERE rating <= 2.0
GROUP BY userId
HAVING SIZE(COLLECT_SET(movieId)) > 0
""").persist(StorageLevel.MEMORY_AND_DISK)

liked_df.count()
disliked_df.count()

liked_df.createOrReplaceTempView("user_liked")
disliked_df.createOrReplaceTempView("user_disliked")

set_sizes_df = spark.sql("""
SELECT
    set_type,
    COUNT(*) AS users_count,
    AVG(set_size) AS avg_set_size,
    percentile_approx(set_size, 0.5) AS median_set_size,
    MIN(set_size) AS min_set_size,
    MAX(set_size) AS max_set_size
FROM (
    SELECT
        'liked_rating_ge_4' AS set_type,
        set_size
    FROM user_liked
    UNION ALL
    SELECT
        'disliked_rating_le_2' AS set_type,
        set_size
    FROM user_disliked
)
GROUP BY set_type
ORDER BY set_type
""")

show_and_save(set_sizes_df, "13_lsh_input_set_sizes")

pair_space_df = spark.sql("""
SELECT
    set_type,
    users_count,
    users_count * (users_count - 1) / 2 AS all_possible_pairs
FROM (
    SELECT
        'liked_rating_ge_4' AS set_type,
        COUNT(*) AS users_count
    FROM user_liked
    UNION ALL
    SELECT
        'disliked_rating_le_2' AS set_type,
        COUNT(*) AS users_count
    FROM user_disliked
)
""")

show_and_save(pair_space_df, "14_lsh_pair_space")

## 9. MinHash i bandy LSH

In [ ]:
random.seed(SEED)

hash_params = [
    (
        random.randint(1, PRIME - 1),
        random.randint(0, PRIME - 1)
    )
    for _ in range(SIGNATURE_LENGTH)
]

hash_params_bc = sc.broadcast(hash_params)


def compute_minhash_signature(movie_set):
    if movie_set is None or len(movie_set) == 0:
        return tuple()

    params = hash_params_bc.value
    signature = []

    for a, b in params:
        min_hash = PRIME
        for movie_id in movie_set:
            h = (a * movie_id + b) % PRIME
            if h < min_hash:
                min_hash = h
        signature.append(int(min_hash))

    return tuple(signature)


def create_user_movies_rdd(df):
    return (
        df.rdd
        .map(lambda r: (
            int(r["userId"]),
            frozenset(int(x) for x in r["movies"])
        ))
        .mapValues(lambda s: (s, len(s)))
        .partitionBy(NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )


user_liked_movies = create_user_movies_rdd(liked_df)
user_disliked_movies = create_user_movies_rdd(disliked_df)


def create_signatures(user_movies, label):
    start = time.time()

    signatures = (
        user_movies
        .map(lambda x: (
            x[0],
            x[1][0],
            x[1][1],
            compute_minhash_signature(x[1][0])
        ))
        .filter(lambda x: len(x[3]) == SIGNATURE_LENGTH)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    print(label, "signatures:", signatures.count())
    print(label, "signature time [min]:", round((time.time() - start) / 60, 2))

    return signatures


def signature_to_bands(record):
    user_id, movie_set, set_size, signature = record

    for band_id in range(NUM_BANDS):
        start = band_id * ROWS_PER_BAND
        end = start + ROWS_PER_BAND
        band_values = signature[start:end]
        yield ((band_id, band_values), user_id)


def create_bands(signatures, label):
    bands = (
        signatures
        .flatMap(signature_to_bands)
        .partitionBy(NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    print(label, "band rows:", bands.count())

    return bands


liked_signatures = create_signatures(user_liked_movies, "liked")
disliked_signatures = create_signatures(user_disliked_movies, "disliked")

liked_bands = create_bands(liked_signatures, "liked")
disliked_bands = create_bands(disliked_signatures, "disliked")

## 10. Kandydaci LSH

In [ ]:
def create_lsh_candidates(bands, label):
    start = time.time()

    bucket_users = (
        bands
        .groupByKey(numPartitions=NUM_PARTITIONS)
        .mapValues(lambda users: sorted(set(users)))
        .filter(lambda x: len(x[1]) >= 2)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    bucket_stats = (
        bucket_users
        .map(lambda x: len(x[1]))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    bucket_count = bucket_stats.count()

    if bucket_count == 0:
        stats_row = [(label, 0, 0, 0.0, 0)]
    else:
        stats = bucket_stats.stats()
        stats_row = [(
            label,
            int(stats.count()),
            int(stats.min()),
            float(stats.mean()),
            int(stats.max())
        )]

    bucket_stats_df = spark.createDataFrame(
        stats_row,
        ["set_type", "nontrivial_buckets", "min_bucket_size", "avg_bucket_size", "max_bucket_size"]
    )

    show_and_save(bucket_stats_df, f"15_{label}_bucket_stats")

    def make_candidate_pairs(bucket_record):
        bucket_key, users = bucket_record
        for i in range(len(users)):
            for j in range(i + 1, len(users)):
                yield (users[i], users[j])

    candidates = (
        bucket_users
        .flatMap(make_candidate_pairs)
        .distinct(NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    candidate_count = candidates.count()

    user_count = (
        bands
        .map(lambda x: x[1])
        .distinct(NUM_PARTITIONS)
        .count()
    )

    all_possible_pairs = user_count * (user_count - 1) / 2

    if all_possible_pairs > 0:
        reduction_pct = 100.0 * (1.0 - candidate_count / all_possible_pairs)
    else:
        reduction_pct = 0.0

    reduction_df = spark.createDataFrame(
        [(
            label,
            int(user_count),
            int(all_possible_pairs),
            int(candidate_count),
            float(reduction_pct)
        )],
        ["set_type", "users_count", "all_possible_pairs", "lsh_candidates", "reduction_pct"]
    )

    show_and_save(reduction_df, f"16_{label}_lsh_reduction")

    print(label, "candidate time [min]:", round((time.time() - start) / 60, 2))

    bucket_stats.unpersist(blocking=False)
    bucket_users.unpersist(blocking=False)

    return candidates


liked_candidates = create_lsh_candidates(liked_bands, "liked")
disliked_candidates = create_lsh_candidates(disliked_bands, "disliked")

liked_bands.unpersist(blocking=False)
disliked_bands.unpersist(blocking=False)
liked_signatures.unpersist(blocking=False)
disliked_signatures.unpersist(blocking=False)
hash_params_bc.destroy()

## 11. Dokładny Jaccard dla kandydatów

In [ ]:
def create_jaccard_scores(candidates, user_movies, label):
    start = time.time()

    user_sets = (
        user_movies
        .partitionBy(NUM_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    joined_first = (
        candidates
        .map(lambda pair: (pair[0], pair))
        .partitionBy(NUM_PARTITIONS)
        .join(user_sets, numPartitions=NUM_PARTITIONS)
    )

    joined_first_by_second = (
        joined_first
        .map(lambda x: (
            x[1][0][1],
            (x[1][0][0], x[1][0][1], x[1][1][0], x[1][1][1])
        ))
        .partitionBy(NUM_PARTITIONS)
    )

    joined_second = (
        joined_first_by_second
        .join(user_sets, numPartitions=NUM_PARTITIONS)
        .map(lambda x: (
            x[1][0][0],
            x[1][0][1],
            x[1][0][2],
            x[1][0][3],
            x[1][1][0],
            x[1][1][1]
        ))
    )

    def score_pair(record):
        user_id1, user_id2, set1, size1, set2, size2 = record

        if size1 <= size2:
            intersection_size = sum(1 for movie_id in set1 if movie_id in set2)
        else:
            intersection_size = sum(1 for movie_id in set2 if movie_id in set1)

        union_size = size1 + size2 - intersection_size

        if union_size == 0:
            jaccard_similarity = 0.0
        else:
            jaccard_similarity = intersection_size / union_size

        return (
            (user_id1, user_id2),
            (intersection_size, union_size, jaccard_similarity)
        )

    scores = (
        joined_second
        .map(score_pair)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    pairs_count = scores.count()

    top30 = (
        scores
        .takeOrdered(30, key=lambda x: (-x[1][2], x[0][0], x[0][1]))
    )

    top30_rows = [
        (
            int(pair[0]),
            int(pair[1]),
            int(values[0]),
            int(values[1]),
            float(values[2])
        )
        for pair, values in top30
    ]

    top30_df = spark.createDataFrame(
        top30_rows,
        ["u1", "u2", "intersection_size", "union_size", "jaccard"]
    )

    show_and_save(top30_df, f"17_{label}_top30_pairs_by_jaccard", n=30)

    print(label, "jaccard pairs:", pairs_count)
    print(label, "jaccard time [min]:", round((time.time() - start) / 60, 2))

    user_sets.unpersist(blocking=False)

    return scores


liked_scores = create_jaccard_scores(liked_candidates, user_liked_movies, "liked")
disliked_scores = create_jaccard_scores(disliked_candidates, user_disliked_movies, "disliked")

## 12. Wpływ thresholda i przecięcie kandydatów liked/disliked

In [ ]:
def count_accepted_by_threshold(scores):
    initial = {threshold: 0 for threshold in THRESHOLDS}

    def seq_op(acc, record):
        jaccard = record[1][2]
        for threshold in THRESHOLDS:
            if jaccard >= threshold:
                acc[threshold] += 1
        return acc

    def comb_op(acc1, acc2):
        return {
            threshold: acc1[threshold] + acc2[threshold]
            for threshold in THRESHOLDS
        }

    return scores.aggregate(initial, seq_op, comb_op)


liked_pair_scores = (
    liked_scores
    .map(lambda x: (x[0], x[1][2]))
    .partitionBy(NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

disliked_pair_scores = (
    disliked_scores
    .map(lambda x: (x[0], x[1][2]))
    .partitionBy(NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

common_liked_disliked_scores = (
    liked_pair_scores
    .join(disliked_pair_scores, numPartitions=NUM_PARTITIONS)
    .persist(StorageLevel.MEMORY_AND_DISK)
)


def count_common_by_threshold(common_scores):
    initial = {threshold: 0 for threshold in THRESHOLDS}

    def seq_op(acc, record):
        liked_jaccard = record[1][0]
        disliked_jaccard = record[1][1]
        for threshold in THRESHOLDS:
            if liked_jaccard >= threshold and disliked_jaccard >= threshold:
                acc[threshold] += 1
        return acc

    def comb_op(acc1, acc2):
        return {
            threshold: acc1[threshold] + acc2[threshold]
            for threshold in THRESHOLDS
        }

    return common_scores.aggregate(initial, seq_op, comb_op)


liked_counts = count_accepted_by_threshold(liked_scores)
disliked_counts = count_accepted_by_threshold(disliked_scores)
common_counts = count_common_by_threshold(common_liked_disliked_scores)

threshold_rows = []

for threshold in THRESHOLDS:
    liked_accepted = liked_counts[threshold]
    disliked_accepted = disliked_counts[threshold]
    common_accepted = common_counts[threshold]
    union_size = liked_accepted + disliked_accepted - common_accepted

    if union_size > 0:
        intersection_pct_of_union = 100.0 * common_accepted / union_size
    else:
        intersection_pct_of_union = 0.0

    if liked_accepted > 0:
        intersection_pct_of_liked = 100.0 * common_accepted / liked_accepted
    else:
        intersection_pct_of_liked = 0.0

    if disliked_accepted > 0:
        intersection_pct_of_disliked = 100.0 * common_accepted / disliked_accepted
    else:
        intersection_pct_of_disliked = 0.0

    threshold_rows.append((
        float(threshold),
        int(liked_accepted),
        int(disliked_accepted),
        int(common_accepted),
        int(union_size),
        float(intersection_pct_of_union),
        float(intersection_pct_of_liked),
        float(intersection_pct_of_disliked)
    ))

threshold_analysis_df = spark.createDataFrame(
    threshold_rows,
    [
        "threshold",
        "liked_pairs_count",
        "disliked_pairs_count",
        "intersection_count",
        "union_count",
        "intersection_pct_of_union",
        "intersection_pct_of_liked",
        "intersection_pct_of_disliked"
    ]
)

show_and_save(threshold_analysis_df, "18_lsh_threshold_intersection_analysis")

threshold_pd = threshold_analysis_df.toPandas()

plt.figure(figsize=(10, 5))
plt.plot(
    threshold_pd["threshold"],
    threshold_pd["intersection_pct_of_union"],
    marker="o"
)
plt.title("Wpływ thresholda na przecięcie kandydatów liked/disliked")
plt.xlabel("Threshold Jaccarda")
plt.ylabel("% przecięcia względem unii par")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(
    threshold_pd["threshold"],
    threshold_pd["liked_pairs_count"],
    marker="o",
    label="liked"
)
plt.plot(
    threshold_pd["threshold"],
    threshold_pd["disliked_pairs_count"],
    marker="o",
    label="disliked"
)
plt.title("Liczba par podobnych użytkowników dla różnych thresholdów")
plt.xlabel("Threshold Jaccarda")
plt.ylabel("Liczba par")
plt.legend()
plt.grid(True)
plt.show()

## Czyszczenie pamięci

In [ ]:
liked_pair_scores.unpersist(blocking=False)
disliked_pair_scores.unpersist(blocking=False)
common_liked_disliked_scores.unpersist(blocking=False)
liked_scores.unpersist(blocking=False)
disliked_scores.unpersist(blocking=False)
liked_candidates.unpersist(blocking=False)
disliked_candidates.unpersist(blocking=False)
user_liked_movies.unpersist(blocking=False)
user_disliked_movies.unpersist(blocking=False)
liked_df.unpersist(blocking=False)
disliked_df.unpersist(blocking=False)
filtered_ratings_df.unpersist(blocking=False)
ratings_df.unpersist(blocking=False)
movies_df.unpersist(blocking=False)

spark.catalog.clearCache()

print("Wyniki zapisane w:", OUTPUT_PATH)